# 02 - Prepare Official Sequence Data

Chuẩn bị dữ liệu official sequence cho GR00T Strong Official-Mini:

- `states_[split].npy`: `[N, S, 44]`, mặc định `S=1`.
- `actions_[split]_chunk.npy`: `[N, H, 44]`, mặc định `H=16`.
- `action_mask_[split].npy`: `[N, H, 44]`.
- `video_frame_manifest_[split].parquet` cho frozen VLM encode.
- `normalization_stats.json` để Notebook 03 normalize và Notebook 04 denormalize metrics.

Notebook chạy CPU. Bật `SMOKE_TEST=True` để debug nhanh.

In [1]:
!pip install -q pandas pyarrow numpy tqdm

In [2]:
from pathlib import Path
import json, random, re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SMOKE_TEST = False
MAX_EPISODES_PER_SUBSET = 5 if SMOKE_TEST else None
MAX_SAMPLES_PER_SPLIT = None

BATCH_ID = "batch01"
REQUIRED_SUBSETS = [
    "gr1_arms_waist.PlaceMilkToMicrowave",
    "gr1_arms_waist.PotatoToMicrowave",
    "gr1_arms_waist.PlateToBowl",
    "gr1_arms_waist.PlateToPlate",
    "gr1_arms_waist.TrayToPlate"
]
DEFAULT_SUBSETS = list(REQUIRED_SUBSETS)
REQUIRE_VIDEO = True
SAMPLE_STRIDE = 4
MASK_DTYPE = np.uint8

def _add_subset(merged, seen, subset, source):
    subset = str(subset).strip()
    if subset.startswith("gr1_") and subset not in seen:
        merged.append(subset)
        seen.add(subset)
        print("Loaded subset:", subset, "from", source)

def load_subset_plan(default):
    merged = []
    seen = set()
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for p in base.rglob("selected_subsets_for_notebook02.json"):
            try:
                obj = json.loads(p.read_text(encoding="utf-8"))
                subsets = obj.get("pipeline_safe_subsets") or obj.get("subsets") or obj
                for subset in [str(x) for x in subsets if str(x).startswith("gr1_")]:
                    _add_subset(merged, seen, subset, p)
            except Exception as exc:
                print("WARN cannot read subset plan", p, exc)
        for p in base.rglob("download_report*.json"):
            try:
                obj = json.loads(p.read_text(encoding="utf-8"))
                subset = obj.get("subset_name")
                ok = obj.get("status") == "downloaded" or (obj.get("has_data_dir") and obj.get("has_meta_dir"))
                if subset and ok:
                    _add_subset(merged, seen, subset, p)
            except Exception as exc:
                print("WARN cannot read download report", p, exc)
        for p in base.rglob("downloaded_subsets.txt"):
            try:
                for line in p.read_text(encoding="utf-8").splitlines():
                    _add_subset(merged, seen, line, p)
            except Exception as exc:
                print("WARN cannot read downloaded_subsets", p, exc)
    required = set(REQUIRED_SUBSETS)
    missing = sorted(required - seen)
    extra = sorted(seen - required)
    gate_report = {"batch_id": BATCH_ID, "required_count": len(REQUIRED_SUBSETS), "loaded_count": len(seen & required), "missing": missing, "extra": extra, "loaded_subsets": merged}
    print("SUBSET_GATE_REPORT", json.dumps(gate_report, indent=2))
    if missing:
        raise RuntimeError(f"Notebook 02 {BATCH_ID} gate failed: missing required subsets: {missing}")
    return [s for s in REQUIRED_SUBSETS if s in seen]

SUBSETS = load_subset_plan(DEFAULT_SUBSETS)
TRAIN_RATIO = 0.8
STATE_HORIZON = 1
ACTION_HORIZON = 16
STATE_DIM = ACTION_DIM = 44
CAMERA_KEY = "observation.images.ego_view"
OUTPUT_ROOT = Path(f"/kaggle/working/gr00t_prepared_official_H{ACTION_HORIZON}_{BATCH_ID}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Loaded subset: gr1_arms_waist.PlateToBowl from /kaggle/input/notebooks/kimthanh211005/notebook08f202570b/selected_subsets_for_notebook02.json
Loaded subset: gr1_arms_waist.PlaceMilkToMicrowave from /kaggle/input/notebooks/kimthanh211005/notebookaa6acea81f/selected_subsets_for_notebook02.json
Loaded subset: gr1_arms_waist.TrayToPlate from /kaggle/input/notebooks/kimthanh211005/notebookf72eb308f0/selected_subsets_for_notebook02.json
Loaded subset: gr1_arms_waist.PlateToPlate from /kaggle/input/notebooks/kimthanh211005/notebook02c0a92f12/selected_subsets_for_notebook02.json
Loaded subset: gr1_arms_waist.PotatoToMicrowave from /kaggle/input/notebooks/kimthanh211005/notebook483d0d4ac7/selected_subsets_for_notebook02.json
SUBSET_GATE_REPORT {
  "batch_id": "batch01",
  "required_count": 5,
  "loaded_count": 5,
  "missing": [],
  "extra": [],
  "loaded_subsets": [
    "gr1_arms_waist.PlateToBowl",
    "gr1_arms_waist.PlaceMilkToMicrowave",
    "gr1_arms_waist.TrayToPlate",
    "gr1_arms_waist

In [3]:
def parse_episode_id(path: Path):
    m = re.search(r"episode[_-](\d+)", path.stem)
    return int(m.group(1)) if m else None

def roots():
    out = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if base.exists():
            for p in base.rglob("gr00t_x_embodiment_sim*"):
                if p.is_dir() and any((p / s).exists() for s in SUBSETS):
                    out.append(p)
    out = sorted(set(out), key=str)
    if not out:
        raise FileNotFoundError("Không tìm thấy gr00t_x_embodiment_sim*. Add Input các output download.")
    return out

def collect_sources():
    src = {s: {"data": None, "videos": []} for s in SUBSETS}
    for r in roots():
        for s in SUBSETS:
            p = r / s
            if not p.exists():
                continue
            if (p / "data").exists() and (p / "meta").exists() and src[s]["data"] is None:
                src[s]["data"] = p
            if (p / "videos").exists():
                src[s]["videos"].append(p)
    missing = [s for s, v in src.items() if v["data"] is None]
    if missing:
        raise FileNotFoundError(f"Thiếu data/meta: {missing}")
    return src

def video_index(video_roots):
    idx = {}
    for root in video_roots:
        vids = list((root / "videos").rglob(f"{CAMERA_KEY}/*.mp4")) or list((root / "videos").rglob("*.mp4"))
        for v in vids:
            ep = parse_episode_id(v)
            if ep is not None and ep not in idx:
                idx[ep] = str(v)
    return idx

def load_tasks(meta):
    tasks = {}
    for name in ["tasks.jsonl", "tasks.json"]:
        p = meta / name
        if not p.exists():
            continue
        try:
            if p.suffix == ".jsonl":
                rows = [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()]
            else:
                rows = json.loads(p.read_text(encoding="utf-8"))
            for obj in rows:
                idx = obj.get("task_index", obj.get("index"))
                txt = obj.get("task", obj.get("text", obj.get("description")))
                if idx is not None and txt is not None:
                    tasks[int(idx)] = str(txt)
        except Exception as exc:
            print("WARN tasks:", p, exc)
    return tasks

def vec(x, dim, col):
    if isinstance(x, np.ndarray):
        a = x.astype(np.float32, copy=False)
    elif isinstance(x, (list, tuple)):
        a = np.asarray(x, np.float32)
    elif isinstance(x, str):
        try:
            a = np.asarray(json.loads(x), np.float32)
        except Exception:
            a = np.fromstring(x.strip("[]"), sep=",", dtype=np.float32)
    else:
        a = np.asarray(x, np.float32)
    a = a.reshape(-1)
    if a.size != dim:
        raise ValueError(f"{col} dim mismatch: expected={dim}, got={a.size}")
    return a

def order_col(df):
    for c in ["frame_index", "timestamp", "index"]:
        if c in df.columns:
            return c
    return df.columns[0]

def ep_id(df, path):
    if "episode_index" in df.columns and len(df):
        return int(df["episode_index"].iloc[0])
    ep = parse_episode_id(path)
    return int(ep if ep is not None else -1)

def task_text(row, tasks):
    for c in ["annotation.human.action.task_description", "task_index"]:
        if c in row.index:
            v = row[c]
            if isinstance(v, str):
                return v
            try:
                return tasks.get(int(v), str(v))
            except Exception:
                return str(v)
    return ""

sources = collect_sources()
for s, v in sources.items():
    print(s, "data=", v["data"], "video_roots=", len(v["videos"]))

gr1_arms_waist.PlaceMilkToMicrowave data= /kaggle/input/notebooks/kimthanh211005/notebookaa6acea81f/gr00t_x_embodiment_sim/gr1_arms_waist.PlaceMilkToMicrowave video_roots= 1
gr1_arms_waist.PotatoToMicrowave data= /kaggle/input/notebooks/kimthanh211005/notebook483d0d4ac7/gr00t_x_embodiment_sim/gr1_arms_waist.PotatoToMicrowave video_roots= 1
gr1_arms_waist.PlateToBowl data= /kaggle/input/notebooks/kimthanh211005/notebook08f202570b/gr00t_x_embodiment_sim/gr1_arms_waist.PlateToBowl video_roots= 1
gr1_arms_waist.PlateToPlate data= /kaggle/input/notebooks/kimthanh211005/notebook02c0a92f12/gr00t_x_embodiment_sim/gr1_arms_waist.PlateToPlate video_roots= 1
gr1_arms_waist.TrayToPlate data= /kaggle/input/notebooks/kimthanh211005/notebookf72eb308f0/gr00t_x_embodiment_sim/gr1_arms_waist.TrayToPlate video_roots= 1


In [4]:
# Pass 1: scan episode length và split theo episode.
episodes, scan_rows = [], []
for subset in SUBSETS:
    root = sources[subset]["data"]
    vids = video_index(sources[subset]["videos"])
    files = sorted((root / "data").rglob("*.parquet"))
    if MAX_EPISODES_PER_SUBSET is not None:
        files = files[:MAX_EPISODES_PER_SUBSET]
    for path in tqdm(files, desc=f"scan {subset}"):
        try:
            df = pd.read_parquet(path)
            if "observation.state" not in df.columns or "action" not in df.columns:
                scan_rows.append({"subset": subset, "path": str(path), "status": "missing_cols"})
                continue
            df = df.sort_values(order_col(df)).reset_index(drop=True)
            ep = ep_id(df, path)
            has_video = ep in vids
            valid_raw = max(0, len(df) - (STATE_HORIZON - 1) - ACTION_HORIZON + 1)
            valid = 0 if (REQUIRE_VIDEO and not has_video) else ((valid_raw + SAMPLE_STRIDE - 1) // SAMPLE_STRIDE)
            episodes.append({"subset": subset, "path": str(path), "episode_index": ep, "length": len(df),
                             "valid_samples": int(valid), "valid_raw_samples": int(valid_raw), "sample_stride": SAMPLE_STRIDE,
                             "video_path": vids.get(ep, ""), "has_video": has_video})
            scan_rows.append({"subset": subset, "path": str(path), "status": "ok", "episode_index": ep,
                              "length": len(df), "valid_samples": int(valid), "valid_raw_samples": int(valid_raw),
                              "sample_stride": SAMPLE_STRIDE, "has_video": has_video})
        except Exception as exc:
            scan_rows.append({"subset": subset, "path": str(path), "status": "error", "error": str(exc)})

episodes_df = pd.DataFrame(episodes)
if episodes_df.empty:
    raise RuntimeError("Không có episode nào hợp lệ.")

train_keys, test_keys = set(), set()
for subset, g in episodes_df[episodes_df.valid_samples > 0].groupby("subset"):
    eps = list(g.episode_index)
    # Dung seed on dinh theo ten subset, khong dung hash() vi Python randomize hash moi session.
    subset_seed = sum((i + 1) * ord(ch) for i, ch in enumerate(str(subset)))
    rng = random.Random(SEED + subset_seed % 10000)
    rng.shuffle(eps)
    n = max(1, int(len(eps) * TRAIN_RATIO)) if len(eps) > 1 else len(eps)
    train_keys.update((subset, int(e)) for e in eps[:n])
    test_keys.update((subset, int(e)) for e in eps[n:])

counts = {"train": 0, "test": 0}
for _, r in episodes_df.iterrows():
    k = (r.subset, int(r.episode_index))
    if r.valid_samples <= 0:
        continue
    if k in train_keys:
        counts["train"] += int(r.valid_samples)
    elif k in test_keys:
        counts["test"] += int(r.valid_samples)
if MAX_SAMPLES_PER_SPLIT is not None:
    counts = {k: min(v, int(MAX_SAMPLES_PER_SPLIT)) for k, v in counts.items()}
print("counts:", counts)
display(episodes_df.groupby("subset").agg(episodes=("episode_index", "count"), valid_samples=("valid_samples", "sum"), video_episodes=("has_video", "sum")))

scan gr1_arms_waist.PlaceMilkToMicrowave:   0%|          | 0/10074 [00:00<?, ?it/s]

scan gr1_arms_waist.PotatoToMicrowave:   0%|          | 0/10053 [00:00<?, ?it/s]

scan gr1_arms_waist.PlateToBowl:   0%|          | 0/10075 [00:00<?, ?it/s]

scan gr1_arms_waist.PlateToPlate:   0%|          | 0/10053 [00:00<?, ?it/s]

scan gr1_arms_waist.TrayToPlate:   0%|          | 0/10074 [00:00<?, ?it/s]

counts: {'train': 886742, 'test': 222413}


,episodes,valid_samples,video_episodes
subset,,,
gr1_arms_waist.PlaceMilkToMicrowave,10074,274456,3000
gr1_arms_waist.PlateToBowl,10075,241529,5000
gr1_arms_waist.PlateToPlate,10053,135973,3000
gr1_arms_waist.PotatoToMicrowave,10053,321032,3000
gr1_arms_waist.TrayToPlate,10074,136165,3000


In [5]:
# Pass 2: ghi memmap .npy và parquet theo batch để tiết kiệm RAM.
arrays = {
    "states_train": np.lib.format.open_memmap(OUTPUT_ROOT / "states_train.npy", "w+", np.float32, (counts["train"], STATE_HORIZON, STATE_DIM)),
    "states_test": np.lib.format.open_memmap(OUTPUT_ROOT / "states_test.npy", "w+", np.float32, (counts["test"], STATE_HORIZON, STATE_DIM)),
    "actions_train": np.lib.format.open_memmap(OUTPUT_ROOT / "actions_train_chunk.npy", "w+", np.float32, (counts["train"], ACTION_HORIZON, ACTION_DIM)),
    "actions_test": np.lib.format.open_memmap(OUTPUT_ROOT / "actions_test_chunk.npy", "w+", np.float32, (counts["test"], ACTION_HORIZON, ACTION_DIM)),
    "mask_train": np.lib.format.open_memmap(OUTPUT_ROOT / "action_mask_train.npy", "w+", MASK_DTYPE, (counts["train"], ACTION_HORIZON, ACTION_DIM)),
    "mask_test": np.lib.format.open_memmap(OUTPUT_ROOT / "action_mask_test.npy", "w+", MASK_DTYPE, (counts["test"], ACTION_HORIZON, ACTION_DIM)),
}
writers, batch_rows = {}, {}

def write_batch(name, rows):
    if not rows:
        return
    table = pa.Table.from_pandas(pd.DataFrame(rows), preserve_index=False)
    path = OUTPUT_ROOT / f"{name}.parquet"
    if name not in writers:
        writers[name] = pq.ParquetWriter(path, table.schema, compression="zstd")
    writers[name].write_table(table)

stat = {"ss": np.zeros(STATE_DIM), "ss2": np.zeros(STATE_DIM), "sn": 0,
        "as": np.zeros(ACTION_DIM), "as2": np.zeros(ACTION_DIM), "an": 0}
idx = {"train": 0, "test": 0}
BATCH = 100000

for _, er in tqdm(episodes_df.iterrows(), total=len(episodes_df), desc="fill"):
    if er.valid_samples <= 0:
        continue
    split = "train" if (er.subset, int(er.episode_index)) in train_keys else "test"
    if idx[split] >= counts[split]:
        continue
    root = sources[er.subset]["data"]
    tasks = load_tasks(root / "meta")
    # Doc parquet mot lan roi sort de giam I/O tren Kaggle.
    df = pd.read_parquet(er.path)
    df = df.sort_values(order_col(df)).reset_index(drop=True)
    states = np.stack([vec(x, STATE_DIM, "state") for x in df["observation.state"].to_list()]).astype(np.float32)
    actions = np.stack([vec(x, ACTION_DIM, "action") for x in df["action"].to_list()]).astype(np.float32)
    oc = order_col(df)
    n = min(int(er.valid_samples), counts[split] - idx[split])
    for sample_i in range(n):
        local = sample_i * SAMPLE_STRIDE
        base = local + STATE_HORIZON - 1
        sid = idx[split]
        sh = states[base - STATE_HORIZON + 1:base + 1]
        ac = actions[base:base + ACTION_HORIZON]
        arrays[f"states_{split}"][sid] = sh
        arrays[f"actions_{split}"][sid] = ac
        arrays[f"mask_{split}"][sid] = 1.0
        if split == "train":
            stat["ss"] += sh.reshape(-1, STATE_DIM).sum(0); stat["ss2"] += (sh.reshape(-1, STATE_DIM) ** 2).sum(0); stat["sn"] += sh.shape[0]
            stat["as"] += ac.sum(0); stat["as2"] += (ac ** 2).sum(0); stat["an"] += ac.shape[0]
        frame = df[oc].iloc[base] if oc in df.columns else base
        rec = {"sample_id": int(sid), "subset": er.subset, "episode_index": int(er.episode_index),
               "frame_index": int(frame) if pd.notna(frame) else int(base),
               "timestamp": float(df["timestamp"].iloc[base]) if "timestamp" in df.columns else float(base),
               "state_horizon": STATE_HORIZON, "action_horizon": ACTION_HORIZON, "sample_stride": SAMPLE_STRIDE,
               "action_start": int(base), "action_end_exclusive": int(base + ACTION_HORIZON),
               "has_video": bool(er.has_video), "task_text": task_text(df.iloc[base], tasks)}
        man = {**rec, "video_path": str(er.video_path) if er.has_video else "", "camera_key": CAMERA_KEY, "split": split}
        for name, row in [(f"{split}_samples", rec), (f"video_frame_manifest_{split}", man)]:
            batch_rows.setdefault(name, []).append(row)
            if len(batch_rows[name]) >= BATCH:
                write_batch(name, batch_rows[name]); batch_rows[name] = []
        idx[split] += 1

for name, rows in batch_rows.items():
    write_batch(name, rows)
for w in writers.values():
    w.close()
for a in arrays.values():
    a.flush()
print("filled:", idx)

fill:   0%|          | 0/50329 [00:00<?, ?it/s]

filled: {'train': 886742, 'test': 222413}


In [6]:
def mean_std(s, s2, n):
    mean = s / max(n, 1)
    var = s2 / max(n, 1) - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-12))
    std = np.where(std < 1e-6, 1.0, std)
    return mean.tolist(), std.tolist()

sm, ss = mean_std(stat["ss"], stat["ss2"], stat["sn"])
am, ast = mean_std(stat["as"], stat["as2"], stat["an"])
norm = {"state_mean": sm, "state_std": ss, "action_mean": am, "action_std": ast,
        "state_n": int(stat["sn"]), "action_n": int(stat["an"]), "space": "raw_train_split"}
(OUTPUT_ROOT / "normalization_stats.json").write_text(json.dumps(norm, indent=2), encoding="utf-8")
(OUTPUT_ROOT / "dataset_scan_report.json").write_text(json.dumps(scan_rows, indent=2), encoding="utf-8")
pd.DataFrame(scan_rows).to_parquet(OUTPUT_ROOT / "dataset_scan_report.parquet", index=False)
episodes_df.to_parquet(OUTPUT_ROOT / "episodes_report.parquet", index=False)
report = {"status": "completed", "batch_id": BATCH_ID, "subsets": SUBSETS,
          "state_horizon": STATE_HORIZON, "action_horizon": ACTION_HORIZON, "sample_stride": SAMPLE_STRIDE,
          "require_video": REQUIRE_VIDEO, "mask_dtype": str(np.dtype(MASK_DTYPE)),
          "state_dim": STATE_DIM, "action_dim": ACTION_DIM, "train_samples": int(idx["train"]),
          "test_samples": int(idx["test"]), "split_by_episode": True, "output_root": str(OUTPUT_ROOT),
          "smoke_test": SMOKE_TEST, "files": sorted(p.name for p in OUTPUT_ROOT.iterdir())}
(OUTPUT_ROOT / "prepare_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
for p in sorted(OUTPUT_ROOT.iterdir()):
    print("-", p.name, round(p.stat().st_size / (1024 ** 2), 3), "MB")

{
  "status": "completed",
  "batch_id": "batch01",
  "subsets": [
    "gr1_arms_waist.PlaceMilkToMicrowave",
    "gr1_arms_waist.PotatoToMicrowave",
    "gr1_arms_waist.PlateToBowl",
    "gr1_arms_waist.PlateToPlate",
    "gr1_arms_waist.TrayToPlate"
  ],
  "state_horizon": 1,
  "action_horizon": 16,
  "sample_stride": 4,
  "require_video": true,
  "mask_dtype": "uint8",
  "state_dim": 44,
  "action_dim": 44,
  "train_samples": 886742,
  "test_samples": 222413,
  "split_by_episode": true,
  "output_root": "/kaggle/working/gr00t_prepared_official_H16_batch01",
  "smoke_test": false,
  "files": [
    "action_mask_test.npy",
    "action_mask_train.npy",
    "actions_test_chunk.npy",
    "actions_train_chunk.npy",
    "dataset_scan_report.json",
    "dataset_scan_report.parquet",
    "episodes_report.parquet",
    "normalization_stats.json",
    "states_test.npy",
    "states_train.npy",
    "test_samples.parquet",
    "train_samples.parquet",
    "video_frame_manifest_test.parquet",
    

## Acceptance

- `states_train.npy`: `[N, S, 44]`.
- `actions_train_chunk.npy`: `[N, H, 44]`.
- `action_mask_train.npy`: `[N, H, 44]`.
- Split theo episode.
- Có `video_frame_manifest_train.parquet`, `video_frame_manifest_test.parquet` và `normalization_stats.json`.